# Neural Networks (Cont.)

This file contains additional hyperparameter tuning for our previously created neural network.

In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [6]:
# Data import and feature engineering

data_unencoded = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')
data_encoded = pd.read_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv')

data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]

data = data_encoded

data['Order Date'] = pd.to_datetime(data['Order Date'])
data = data.sort_values(by=['Survey ResponseID', 'Order Date'])
data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)

data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)

data = data.drop(columns=['Order Date'])

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['Survey ResponseID'] = encoder.fit_transform(data['Survey ResponseID'])

from sklearn.preprocessing import StandardScaler

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

train_cats = set(train['Category'].unique())
test_cats = set(test['Category'].unique())
unseen = test_cats - train_cats
if len(unseen) > 0:
    test = test[~test['Category'].isin(unseen)]


data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']
X_train_scaled = StandardScaler().fit_transform(X_train)

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']
X_test_scaled = StandardScaler().fit_transform(X_test)

X_train = X_train.drop(X_train.filter(regex='^Shipping Address').columns, axis=1)
X_test = X_test.drop(X_test.filter(regex='^Shipping Address').columns, axis=1)

X_train = X_train.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])
X_test = X_test.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])

X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

/tmp/ipykernel_2539/424565588.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]
/tmp/ipykernel_2539/424565588.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)
/tmp/ipykernel_2539/424565588.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

In [7]:
# Recreating our best neural network thus far

import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        30,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,825 (878.22 KB)

 Trainable params: 224,825 (878.22 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [11]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

Epoch 1/30


W0000 00:00:1776719777.689627    2539 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.0619 - loss: 6.4446
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0718 - loss: 6.0582
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0769 - loss: 5.9744
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0809 - loss: 5.8920
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0841 - loss: 5.8128
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0866 - loss: 5.7396
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0898 - loss: 5.6727
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0924 - loss: 5.6129
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0950 - loss: 5.5594
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0966 - loss: 5.5114
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.0989 - loss: 5.4687
Epoch 12/30
3552/3552 ━━━━━━━━

In [12]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 3s - 3ms/step - accuracy: 0.0692 - loss: 5.9101

Test accuracy: 0.06919153779745102


Next we will try tuning the optimizer

In [13]:
def build_model():
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)

    model = tf.keras.Sequential()

    model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

    model.add(tf.keras.layers.Dense(300, activation="relu")) 

    model.add(tf.keras.layers.Dense(100, activation="relu"))

    model.add(tf.keras.layers.Dense(1625, activation="softmax")) 
    
    return model

In [14]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)

model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30


W0000 00:00:1776720128.930086    2539 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0495 - loss: 6.5481
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0609 - loss: 6.1450
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0682 - loss: 6.0382
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0728 - loss: 5.9508
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0763 - loss: 5.8694
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0801 - loss: 5.7947
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0838 - loss: 5.7269
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0869 - loss: 5.6661
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0899 - loss: 5.6112
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0921 - loss: 5.5613
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.0944 - loss: 5.5155
Epoch 12/30
3552/3552 ━━━━━━━━

In [15]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30


W0000 00:00:1776720517.223625    2539 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0589 - loss: 6.1924
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0697 - loss: 5.8385
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0787 - loss: 5.6179
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0841 - loss: 5.4671
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0872 - loss: 5.3562
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0912 - loss: 5.2642
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0932 - loss: 5.2024
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - accuracy: 0.0956 - loss: 5.1554
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0983 - loss: 5.1156
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0997 - loss: 5.0885
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.1014 - loss: 5.0593
Epoch 12/30
3552/3552 ━━━━━━━━

In [16]:
optimizer = tf.keras.optimizers.Adagrad(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30


W0000 00:00:1776720987.126768    2539 cpu_allocator_impl.cc:82] Allocation of 45916620 exceeds 10% of free system memory.


3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0441 - loss: 7.1500
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0572 - loss: 6.5410
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0579 - loss: 6.3106
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0582 - loss: 6.2332
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0583 - loss: 6.1953
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0584 - loss: 6.1714
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0586 - loss: 6.1540
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0594 - loss: 6.1402
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0600 - loss: 6.1284
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0604 - loss: 6.1181
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0606 - loss: 6.1087
Epoch 12/30
3552/3552 ━━━━━━━━

In [17]:
optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0664 - loss: 6.1213
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0751 - loss: 5.9125
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0800 - loss: 5.8346
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0831 - loss: 5.7978
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0852 - loss: 5.7864
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0855 - loss: 5.7945
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0855 - loss: 5.8164
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0847 - loss: 5.8200
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0836 - loss: 5.8358
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0852 - loss: 5.8375
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0852 - loss: 5.8470
Epoch 12/30
3552/35

In [18]:
optimizer = tf.keras.optimizers.Adamax(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0546 - loss: 6.2884
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0604 - loss: 6.0271
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0706 - loss: 5.8638
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0767 - loss: 5.7250
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0829 - loss: 5.6088
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0855 - loss: 5.5126
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0892 - loss: 5.4321
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0928 - loss: 5.3642
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0957 - loss: 5.3064
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0986 - loss: 5.2559
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.1011 - loss: 5.2117
Epoch 12/30
3552/35

In [19]:
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.0625 - loss: 6.1136
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0755 - loss: 5.7320
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - accuracy: 0.0837 - loss: 5.4981
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.0893 - loss: 5.3397
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0933 - loss: 5.2319
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0983 - loss: 5.1394
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.1016 - loss: 5.0783
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.1034 - loss: 5.0265
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.1043 - loss: 4.9843
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.1067 - loss: 4.9593
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 21s 5ms/step - accuracy: 0.1081 - loss: 4.9222
Epoch 12/30
3552/35

In [20]:
optimizer = tf.keras.optimizers.AdamW(learning_rate=0.001)
model = build_model()
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.0604 - loss: 6.1750
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0719 - loss: 5.8226
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0779 - loss: 5.5950
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0839 - loss: 5.4462
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0892 - loss: 5.3236
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0928 - loss: 5.2382
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0945 - loss: 5.1647
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0968 - loss: 5.1157
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0992 - loss: 5.0591
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.1010 - loss: 5.0260
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.1014 - loss: 5.0070
Epoch 12/30
3552/35

Next, we will try various regularization techniques, including L1, L2, Dropout, and Max-Norm regularization. This should reduce the overfitting of our training data.

In [21]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l2(0.01))) 

model.add(tf.keras.layers.Dense(100, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l2(0.01)))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

In [22]:
# sgd is still our best performing optimizer

model.compile(loss="sparse_categorical_crossentropy", optimizer='sgd',
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0599 - loss: 11.3116 - val_accuracy: 0.0604 - val_loss: 8.8949
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0643 - loss: 7.7994 - val_accuracy: 0.0612 - val_loss: 7.1201
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0631 - loss: 6.7418 - val_accuracy: 0.0612 - val_loss: 6.5477
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0628 - loss: 6.3861 - val_accuracy: 0.0612 - val_loss: 6.3441
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.0637 - loss: 6.2524 - val_accuracy: 0.0612 - val_loss: 6.2497
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0646 - loss: 6.1955 - val_accuracy: 0.0612 - val_loss: 6.1999
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - accuracy: 0.0653 - loss: 6.1679 - val_accuracy: 0.0613 - val_loss: 6.1743
Epoch 8/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.0654 - loss: 

In [23]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l1(0.01))) 

model.add(tf.keras.layers.Dense(100, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l1(0.01)))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [24]:
model.compile(loss="sparse_categorical_crossentropy", optimizer='sgd',
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0542 - loss: 21.1842 - val_accuracy: 0.0612 - val_loss: 6.8190
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0540 - loss: 6.4628 - val_accuracy: 0.0612 - val_loss: 6.2694
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0544 - loss: 6.3090 - val_accuracy: 0.0612 - val_loss: 6.2007
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0545 - loss: 6.2668 - val_accuracy: 0.0612 - val_loss: 6.1777
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0550 - loss: 6.2475 - val_accuracy: 0.0612 - val_loss: 6.1663
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0550 - loss: 6.2359 - val_accuracy: 0.0612 - val_loss: 6.1595
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0552 - loss: 6.2280 - val_accuracy: 0.0612 - val_loss: 6.1560
Epoch 8/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.0550 - loss: 

In [25]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l1(0.01))) 

model.add(tf.keras.layers.Dropout(rate=0.2))

model.add(tf.keras.layers.Dense(100, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l1(0.01)))

model.add(tf.keras.layers.Dropout(rate=0.2))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 


In [27]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(101, )))

model.add(tf.keras.layers.Dense(300, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_constraint=tf.keras.constraints.max_norm(1.))) 

model.add(tf.keras.layers.Dense(100, activation="relu",
                                kernel_initializer="he_normal",
                                kernel_constraint=tf.keras.constraints.max_norm(1.)))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

In [28]:
model.compile(loss="sparse_categorical_crossentropy", optimizer='sgd',
                  metrics=["accuracy"])

model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2)

test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0591 - loss: 6.5039 - val_accuracy: 0.0597 - val_loss: 6.2692
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0710 - loss: 6.1029 - val_accuracy: 0.0555 - val_loss: 6.1866
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0755 - loss: 6.0173 - val_accuracy: 0.0579 - val_loss: 6.1583
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0782 - loss: 5.9447 - val_accuracy: 0.0600 - val_loss: 6.1487
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0809 - loss: 5.8766 - val_accuracy: 0.0608 - val_loss: 6.1491
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0836 - loss: 5.8143 - val_accuracy: 0.0615 - val_loss: 6.1525
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0857 - loss: 5.7574 - val_accuracy: 0.0619 - val_loss: 6.1552
Epoch 8/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0880 - loss: 5